# Chapter 1: Linear Algebra Foundations for LLMs

This notebook covers the linear algebra concepts underpinning every LLM operation — from embedding lookups to attention.

Every forward pass through a transformer is a sequence of matrix multiplications, vector norms, and tensor contractions. Understanding the linear algebra behind these operations is essential for reading research papers, debugging shapes, and implementing components from scratch.

**Topics covered:**
1. Vectors & Norms
2. Matrix Multiplication
3. Tensors in PyTorch
4. Eigenvalues & Eigenvectors
5. SVD & Low-Rank Approximation (LoRA)
6. Embeddings as Matrix Lookups
7. Scaled Dot-Product Attention
8. LoRA Layer Implementation

**Install:** `!pip install torch` if needed.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
print(f"PyTorch {torch.__version__}")

## 1. Vectors & Norms

A **vector** $\mathbf{x} \in \mathbb{R}^d$ is the fundamental data unit in LLMs — every token is represented as one.

### L2 Norm (Euclidean Norm)

$$\|\mathbf{x}\|_2 = \sqrt{\sum_{i=1}^{d} x_i^2}$$

This measures the "length" of a vector. Normalizing by the L2 norm projects a vector onto the unit hypersphere.

### Dot Product

$$\mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^{d} u_i v_i$$

The dot product measures how much two vectors "align". It is the core operation in attention: each query dot-products with each key.

### Cosine Similarity

$$\cos\theta = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \, \|\mathbf{v}\|}$$

Cosine similarity normalizes for magnitude — it measures **direction** only. Range: $[-1, 1]$. Used heavily in retrieval and embedding comparisons.

**Intuition:** Two embeddings with cosine similarity close to 1 represent semantically similar tokens; close to -1 means opposite meaning; near 0 means unrelated.

In [ ]:
# Create two embedding vectors (e.g., word embeddings of dimension 8)
u = torch.tensor([1.0, 2.0, 3.0, 0.5, -1.0, 0.2, 0.8, -0.3])
v = torch.tensor([0.5, 1.8, 2.9, 0.6, -0.9, 0.3, 0.7, -0.2])

# L2 norms
norm_u = torch.norm(u, p=2)
norm_v = torch.norm(v, p=2)
print(f"||u||_2 = {norm_u:.4f}")
print(f"||v||_2 = {norm_v:.4f}")

# Dot product
dot = torch.dot(u, v)
print(f"u . v   = {dot:.4f}")

# Cosine similarity (manual)
cos_manual = dot / (norm_u * norm_v)
print(f"cos(u,v) manual = {cos_manual:.4f}")

# Cosine similarity using PyTorch (needs 2D input for F.cosine_similarity)
cos_torch = F.cosine_similarity(u.unsqueeze(0), v.unsqueeze(0))
print(f"cos(u,v) torch  = {cos_torch.item():.4f}")

# Normalized vectors (unit vectors)
u_hat = u / norm_u
v_hat = v / norm_v
print(f"\n||u_hat||_2 = {torch.norm(u_hat):.4f}  (should be 1.0)")

# L1 norm (for reference)
norm_u_l1 = torch.norm(u, p=1)
print(f"||u||_1 = {norm_u_l1:.4f}")

## 2. Matrix Multiplication

Matrix multiplication is the single most important operation in neural networks. Every linear layer, every attention projection, every feed-forward computation reduces to a `matmul`.

### Definition

For $\mathbf{A} \in \mathbb{R}^{m \times k}$ and $\mathbf{B} \in \mathbb{R}^{k \times n}$:

$$(\mathbf{A}\mathbf{B})_{ij} = \sum_{l=1}^{k} A_{il} B_{lj}$$

**Shape rule:** $(m \times k)(k \times n) \to (m \times n)$. The inner dimensions must match.

### In LLMs

- **Linear layer:** $\mathbf{y} = \mathbf{x}\mathbf{W}^T + \mathbf{b}$ where $\mathbf{W} \in \mathbb{R}^{d_{out} \times d_{in}}$
- **QKV projections:** multiply token matrix $(T \times d)$ by weight $(d \times d_k)$
- **Attention scores:** $(T \times d_k)(d_k \times T) \to (T \times T)$

### Batched Matrix Multiplication

In practice, we process a **batch** of sequences simultaneously. PyTorch's `torch.bmm` and `torch.matmul` support batched matmul:

$$(B, m, k) \otimes (B, k, n) \to (B, m, n)$$

where $B$ is the batch size. Each slice along the batch dimension is multiplied independently.

In [ ]:
# Basic matrix multiplication
A = torch.randn(4, 6)   # (m=4, k=6)
B = torch.randn(6, 8)   # (k=6, n=8)

C1 = torch.matmul(A, B)
C2 = A @ B               # @ is syntactic sugar for matmul
print(f"A shape: {A.shape}")
print(f"B shape: {B.shape}")
print(f"A @ B shape: {C1.shape}  (expected 4x8)")
print(f"Results match: {torch.allclose(C1, C2)}")

# Simulate a linear projection: tokens (T=6, d=32) -> (T=6, d_k=16)
T, d, d_k = 6, 32, 16
X = torch.randn(T, d)        # token matrix
W_q = torch.randn(d, d_k)   # query weight
Q = X @ W_q
print(f"\nToken matrix X: {X.shape}")
print(f"Query weight W_q: {W_q.shape}")
print(f"Query matrix Q: {Q.shape}")

# Batched matrix multiplication: B=2 batches, T=6 tokens, d=32 dims
B_size, T, d, d_k = 2, 6, 32, 16
X_batch = torch.randn(B_size, T, d)      # (B, T, d)
W_q_batch = torch.randn(d, d_k)          # (d, d_k)  -- shared across batch

# Using matmul: broadcasts W_q across batch dimension
Q_batch = X_batch @ W_q_batch            # (B, T, d) @ (d, d_k) -> (B, T, d_k)
print(f"\nBatched: X_batch {X_batch.shape} @ W_q {W_q_batch.shape} = {Q_batch.shape}")

# Explicit batched matmul with per-batch weights
W_q_per_batch = torch.randn(B_size, d, d_k)   # (B, d, d_k)
Q_explicit = torch.bmm(X_batch, W_q_per_batch) # (B, T, d_k)
print(f"torch.bmm result: {Q_explicit.shape}")

## 3. Tensors in PyTorch

LLM computations work with **rank-3 tensors** of shape $(B, T, d)$:

- $B$ — batch size (number of sequences processed in parallel)
- $T$ — sequence length (number of tokens)
- $d$ — model dimension (embedding size)

### Key Tensor Operations

| Operation | Description |
|---|---|
| `reshape(B, T*d)` | Flatten last two dims |
| `permute(0, 2, 1)` | Transpose $T$ and $d$ dims |
| `view(B, T, H, d//H)` | Split $d$ into $H$ heads |
| `contiguous()` | Make memory contiguous after permute |

### Einstein Summation (`einsum`)

`torch.einsum` expresses arbitrary tensor contractions with index notation:

$$\text{einsum}('btd,de\to bte', \mathbf{X}, \mathbf{W})$$

reads as: for each batch $b$, token $t$, output dim $e$ — sum over $d$: $\text{out}_{bte} = \sum_d X_{btd} \cdot W_{de}$.

This is equivalent to `X @ W` when $\mathbf{W} \in \mathbb{R}^{d \times e}$, but `einsum` generalizes to any contraction.

In [ ]:
B, T, d = 2, 6, 32
x = torch.randn(B, T, d)   # rank-3 tensor: (batch, seq_len, dim)
print(f"x.shape = {x.shape}  | ndim={x.ndim} | numel={x.numel()}")

# reshape: collapse T and d into one dimension
x_flat = x.reshape(B, T * d)
print(f"\nreshaped to (B, T*d): {x_flat.shape}")

# permute: move sequence dim to last position -> (B, d, T)
x_perm = x.permute(0, 2, 1)   # (B, d, T)
print(f"permuted to (B, d, T): {x_perm.shape}")

# Multi-head split: split d into H heads of size d_head
H = 4
d_head = d // H
x_mh = x.view(B, T, H, d_head)   # (B, T, H, d_head)
print(f"multi-head view (B, T, H, d_head): {x_mh.shape}")

# Transpose for attention: (B, H, T, d_head)
x_mh_t = x_mh.permute(0, 2, 1, 3).contiguous()
print(f"after permute for attn (B, H, T, d_head): {x_mh_t.shape}")

# einsum: linear projection (B, T, d) x (d, e) -> (B, T, e)
d_out = 64
W = torch.randn(d, d_out)
y_einsum = torch.einsum('btd,de->bte', x, W)
y_matmul = x @ W   # equivalent
print(f"\neinsum result shape: {y_einsum.shape}")
print(f"matmul result shape: {y_matmul.shape}")
print(f"einsum == matmul: {torch.allclose(y_einsum, y_matmul, atol=1e-5)}")

# einsum: attention scores (B, H, T, dk) x (B, H, dk, T) -> (B, H, T, T)
dk = 8
Q = torch.randn(B, H, T, dk)
K = torch.randn(B, H, T, dk)
scores = torch.einsum('bhik,bhjk->bhij', Q, K)   # b=batch, h=head, i=query pos, j=key pos
print(f"\nAttention scores via einsum (B, H, T, T): {scores.shape}")

## 4. Eigenvalues & Eigenvectors

An **eigenvector** of matrix $\mathbf{A}$ is a nonzero vector $\mathbf{v}$ that only gets scaled (not rotated) by $\mathbf{A}$:

$$\mathbf{A}\mathbf{v} = \lambda\mathbf{v}$$

where $\lambda \in \mathbb{C}$ is the corresponding **eigenvalue**.

### Characteristic Equation

$$\det(\mathbf{A} - \lambda\mathbf{I}) = 0$$

Solving this polynomial gives the eigenvalues. For an $n \times n$ matrix, there are exactly $n$ eigenvalues (counting multiplicity, over $\mathbb{C}$).

### Relevance to LLMs

- **Weight initialization:** Eigenvalue analysis guides initialization schemes (e.g., Glorot/He) to keep activations in a stable range
- **Attention analysis:** Researchers inspect eigenvalue spectra of attention weight matrices to understand what information different heads capture
- **Optimization landscape:** The eigenvalues of the Hessian $\nabla^2 \mathcal{L}$ determine convergence speed — large eigenvalues require small learning rates
- **Layer Normalization:** LayerNorm implicitly normalizes the eigenvalue scale of activations

For a **symmetric** matrix $\mathbf{A} = \mathbf{A}^T$, all eigenvalues are real and eigenvectors are orthogonal — this is the spectral theorem.

In [ ]:
# Create a symmetric 3x3 matrix (symmetric => real eigenvalues)
M = torch.tensor([
    [4.0, 2.0, 1.0],
    [2.0, 5.0, 3.0],
    [1.0, 3.0, 6.0]
])

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = torch.linalg.eig(M)
print("Matrix M:")
print(M)
print(f"\nEigenvalues (may be complex): {eigenvalues}")

# For symmetric matrix, use eigh (faster, guaranteed real)
eigenvalues_real, eigenvectors_real = torch.linalg.eigh(M)
print(f"\nEigenvalues (real, from eigh): {eigenvalues_real}")
print(f"Eigenvectors (columns):\n{eigenvectors_real}")

# Verify A*v = lambda*v for each eigenpair
print("\nVerification: A*v vs lambda*v")
for i in range(3):
    v = eigenvectors_real[:, i]
    lam = eigenvalues_real[i]
    Av = M @ v
    lv = lam * v
    residual = torch.norm(Av - lv)
    print(f"  Eigenpair {i}: ||A*v - lambda*v|| = {residual:.2e}  (should be ~0)")

# Eigenvectors of symmetric matrix are orthonormal
print("\nOrthogonality check (V^T V should be identity):")
VtV = eigenvectors_real.T @ eigenvectors_real
print(VtV.round(decimals=4))

# Spectral decomposition: A = V * diag(lambda) * V^T
A_reconstructed = eigenvectors_real @ torch.diag(eigenvalues_real) @ eigenvectors_real.T
print(f"\nReconstruction error: {torch.norm(M - A_reconstructed):.2e}")

## 5. SVD & Low-Rank Approximation

**Singular Value Decomposition (SVD)** factorizes any matrix $\mathbf{A} \in \mathbb{R}^{m \times n}$ as:

$$\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$$

where:
- $\mathbf{U} \in \mathbb{R}^{m \times m}$ — left singular vectors (orthonormal)
- $\boldsymbol{\Sigma} \in \mathbb{R}^{m \times n}$ — diagonal matrix of **singular values** $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$
- $\mathbf{V} \in \mathbb{R}^{n \times n}$ — right singular vectors (orthonormal)

### Rank-$r$ Approximation

Keep only the top-$r$ singular values (Eckart–Young theorem — this is the **optimal** rank-$r$ approximation):

$$\mathbf{A}_r = \sum_{i=1}^{r} \sigma_i \mathbf{u}_i \mathbf{v}_i^T$$

### Connection to LoRA

**LoRA (Low-Rank Adaptation)** exploits the insight that fine-tuning updates $\Delta\mathbf{W}$ have low intrinsic rank. Instead of updating $\mathbf{W}_0 \in \mathbb{R}^{d \times d}$ directly ($d^2$ parameters), LoRA parameterizes:

$$\Delta\mathbf{W} = \mathbf{B}\mathbf{A}, \quad \mathbf{B} \in \mathbb{R}^{d \times r},\ \mathbf{A} \in \mathbb{R}^{r \times d}$$

with $r \ll d$, reducing trainable parameters from $d^2$ to $2rd$. SVD gives us the theoretical justification: if $\Delta\mathbf{W}$ has low effective rank, this factorization captures most of its "information".

In [ ]:
# Create a matrix to decompose
torch.manual_seed(42)
m, n = 10, 8
A = torch.randn(m, n)

# Full SVD
U, S, Vh = torch.linalg.svd(A, full_matrices=False)
print(f"A shape: {A.shape}")
print(f"U shape: {U.shape}  (left singular vectors)")
print(f"S shape: {S.shape}  (singular values)")
print(f"Vh shape: {Vh.shape} (right singular vectors, transposed)")
print(f"\nSingular values: {S.round(decimals=3)}")

# Full reconstruction
A_reconstructed = U @ torch.diag(S) @ Vh
print(f"\nFull reconstruction error: {torch.norm(A - A_reconstructed):.2e}")

# Rank-1 approximation
r = 1
A_r1 = S[0] * U[:, 0:1] @ Vh[0:1, :]   # outer product
err_r1 = torch.norm(A - A_r1)
print(f"\nRank-1 approximation error: {err_r1:.4f}")
print(f"Rank-1 explained variance: {(S[0]**2 / (S**2).sum() * 100):.1f}%")

# Rank-3 approximation
r = 3
A_r3 = U[:, :r] @ torch.diag(S[:r]) @ Vh[:r, :]
err_r3 = torch.norm(A - A_r3)
print(f"Rank-3 approximation error: {err_r3:.4f}")
print(f"Rank-3 explained variance: {(S[:r]**2).sum() / (S**2).sum() * 100:.1f}%")

# LoRA-style decomposition of a weight update
print("\n--- LoRA parameter efficiency ---")
d_in, d_out = 768, 768
for r_lora in [1, 4, 8, 16, 64]:
    params_full = d_in * d_out
    params_lora = r_lora * (d_in + d_out)
    ratio = params_lora / params_full * 100
    print(f"  r={r_lora:3d}: LoRA params = {params_lora:7,d}  vs full = {params_full:,d}  ({ratio:.2f}%)")

## 6. Embeddings as Matrix Lookups

The very first operation in any LLM is an **embedding lookup**. The model maintains an embedding table:

$$\mathbf{E} \in \mathbb{R}^{V \times d}$$

where $V$ is the vocabulary size and $d$ is the embedding dimension. For GPT-3: $V = 50{,}257$, $d = 12{,}288$, making $\mathbf{E}$ a matrix of ~618M parameters — just for embeddings.

### Lookup is a Matrix Row Selection

Given token ID $i$, the embedding is simply row $i$ of $\mathbf{E}$:

$$\mathbf{e}_i = \mathbf{E}[i, :] \in \mathbb{R}^d$$

Equivalently, if $\mathbf{o}_i$ is a one-hot vector, then $\mathbf{e}_i = \mathbf{E}^T \mathbf{o}_i$. The embedding lookup is a matrix multiplication — but we short-circuit to indexing for efficiency.

### Word Analogy Structure

Well-trained embeddings exhibit linear structure:

$$\mathbf{e}_{\text{king}} - \mathbf{e}_{\text{man}} + \mathbf{e}_{\text{woman}} \approx \mathbf{e}_{\text{queen}}$$

This linearity emerges from training — the model learns to encode semantic relationships as directions in the embedding space.

### Tie Weights

Many LLMs (GPT-2, LLaMA) **tie** the input embedding matrix $\mathbf{E}$ with the output projection $\mathbf{E}^T$ (used before the final softmax). This halves the vocabulary-related parameter count and improves performance.

In [ ]:
torch.manual_seed(42)

# Create an embedding table: vocab of 100 tokens, each embedded in 16 dims
vocab_size = 100
embedding_dim = 16
emb = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

print(f"Embedding weight shape: {emb.weight.shape}  (vocab_size x embedding_dim)")
print(f"Total embedding parameters: {emb.weight.numel():,}")

# Look up tokens
token_ids = torch.tensor([5, 42, 17, 99])   # batch of 4 token IDs
embeddings = emb(token_ids)
print(f"\nToken IDs: {token_ids.tolist()}")
print(f"Embeddings shape: {embeddings.shape}  (num_tokens x embedding_dim)")

# Batched sequence lookup: (B=2, T=5) -> (B=2, T=5, d=16)
B, T = 2, 5
token_seq = torch.randint(0, vocab_size, (B, T))
seq_embeddings = emb(token_seq)
print(f"\nToken sequence shape: {token_seq.shape}")
print(f"Sequence embeddings shape: {seq_embeddings.shape}")

# Verify that lookup == row indexing into weight matrix
manual_lookup = emb.weight[token_ids]
print(f"\nDirect row access matches nn.Embedding: {torch.allclose(embeddings, manual_lookup)}")

# Demonstrate word analogy structure with random embeddings
# (In a real model these would be trained)
print("\n--- Analogy demo (random embeddings, for illustration) ---")
idx_king   = 10
idx_man    = 20
idx_woman  = 30
idx_queen  = 40

# The analogy vector
analogy_vec = emb.weight[idx_king] - emb.weight[idx_man] + emb.weight[idx_woman]

# Find closest token by cosine similarity
all_embs = emb.weight   # (V, d)
sims = F.cosine_similarity(analogy_vec.unsqueeze(0), all_embs)  # (V,)
# Exclude source tokens
for i in [idx_king, idx_man, idx_woman]:
    sims[i] = -1.0

top5 = sims.topk(5)
print(f"king - man + woman -> top-5 nearest token IDs: {top5.indices.tolist()}")
print(f"(queen ID={idx_queen} similarity: {sims[idx_queen]:.4f})")
print("Note: With random embeddings this is random; trained embeddings would give 'queen'.")

## 7. Scaled Dot-Product Attention

Attention is the core mechanism of transformers. Given input matrices $\mathbf{Q}$, $\mathbf{K}$, $\mathbf{V}$ (query, key, value):

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\mathbf{V}$$

### Why Scale by $\sqrt{d_k}$?

If $\mathbf{q}$ and $\mathbf{k}$ are random vectors with unit-variance components, then:

$$\text{Var}(\mathbf{q} \cdot \mathbf{k}) = \sum_{i=1}^{d_k} \text{Var}(q_i k_i) = d_k$$

So $\mathbf{q} \cdot \mathbf{k}$ has standard deviation $\sqrt{d_k}$. Without scaling, large $d_k$ pushes dot products into extreme regions of softmax where gradients vanish. Dividing by $\sqrt{d_k}$ restores unit variance.

### Causal Masking

For **autoregressive** generation, token at position $t$ must not attend to positions $t+1, t+2, \ldots$ (future tokens). We apply a **causal mask** by setting those attention logits to $-\infty$ before softmax:

$$\text{mask}_{ij} = \begin{cases} 0 & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

After softmax, $e^{-\infty} = 0$, so masked positions contribute nothing to the output.

### Multi-Head Attention

In practice, we run $H$ attention heads in parallel, each projecting to dimension $d_k = d/H$:

$$\text{MultiHead}(\mathbf{X}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H)\mathbf{W}^O$$

$$\text{head}_i = \text{Attention}(\mathbf{X}\mathbf{W}_i^Q, \mathbf{X}\mathbf{W}_i^K, \mathbf{X}\mathbf{W}_i^V)$$

In [ ]:
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.
    Args:
        Q: (B, H, T, d_k) query
        K: (B, H, T, d_k) key
        V: (B, H, T, d_v) value
        mask: (1, 1, T, T) boolean mask (True = keep, False = mask out)
    Returns:
        output: (B, H, T, d_v)
        attn_weights: (B, H, T, T)
    """
    d_k = Q.shape[-1]

    # Step 1: Compute raw attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1))   # (B, H, T, T)

    # Step 2: Scale
    scores = scores / math.sqrt(d_k)

    # Step 3: Apply causal mask (set future positions to -inf)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # Step 4: Softmax over key dimension
    attn_weights = F.softmax(scores, dim=-1)

    # Step 5: Weighted sum of values
    output = torch.matmul(attn_weights, V)           # (B, H, T, d_v)
    return output, attn_weights


# Test with (B=2, H=4, T=6, d_k=8)
B, H, T, d_k = 2, 4, 6, 8
torch.manual_seed(42)
Q = torch.randn(B, H, T, d_k)
K = torch.randn(B, H, T, d_k)
V = torch.randn(B, H, T, d_k)

# Causal mask: lower triangular
causal_mask = torch.tril(torch.ones(T, T)).bool().unsqueeze(0).unsqueeze(0)  # (1,1,T,T)
print(f"Causal mask (T={T}):")
print(causal_mask.squeeze().int())

# Run attention
output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print(f"\nOutput shape: {output.shape}   (B, H, T, d_k)")
print(f"Attn weights shape: {attn_weights.shape}")

# Verify causal property: for position i, only attends to j<=i
w = attn_weights[0, 0]   # head 0, batch 0: (T, T)
print(f"\nAttention weights (head 0, batch 0):")
print(w.detach().round(decimals=3))
print(f"Upper triangle sum (should be 0): {w.triu(diagonal=1).sum():.6f}")

# Verify rows sum to 1 (softmax property)
print(f"Row sums: {w.sum(dim=-1).detach().round(decimals=4)}  (all should be 1.0)")

# Demonstrate scaling effect
print("\n--- Effect of scaling ---")
for dk_test in [8, 64, 512]:
    q_test = torch.randn(100, dk_test)
    k_test = torch.randn(100, dk_test)
    raw = (q_test * k_test).sum(dim=-1)   # dot products
    print(f"  d_k={dk_test:3d}: raw dot std={raw.std():.2f}, scaled std={raw.std()/math.sqrt(dk_test):.2f}")

## 8. LoRA Layer

**LoRA (Low-Rank Adaptation)**, introduced by Hu et al. (2021), is the dominant PEFT (parameter-efficient fine-tuning) method for LLMs.

### Formulation

For a pretrained weight $\mathbf{W}_0 \in \mathbb{R}^{d_{out} \times d_{in}}$, the modified forward pass is:

$$\mathbf{y} = \mathbf{W}_0\mathbf{x} + \frac{\alpha}{r}\mathbf{B}\mathbf{A}\mathbf{x}$$

where:
- $\mathbf{A} \in \mathbb{R}^{r \times d_{in}}$ — initialized with Gaussian noise
- $\mathbf{B} \in \mathbb{R}^{d_{out} \times r}$ — initialized to **zero** (so $\Delta W = 0$ at start)
- $r$ — rank (hyperparameter, typically 4–64)
- $\alpha$ — scaling factor (often equals $r$, so $\alpha/r = 1$)

### Parameter Count

| Method | Trainable parameters |
|---|---|
| Full fine-tuning | $d_{in} \times d_{out}$ |
| LoRA rank $r$ | $r(d_{in} + d_{out})$ |

For $d_{in} = d_{out} = 4096$, $r = 8$: LoRA uses **0.39%** of parameters of full fine-tuning.

### Merging

At inference time, LoRA weights can be merged back into $\mathbf{W}_0$:

$$\mathbf{W}' = \mathbf{W}_0 + \frac{\alpha}{r}\mathbf{B}\mathbf{A}$$

This adds **zero inference overhead** compared to the original model.

In [ ]:
class LoRALinear(nn.Module):
    """
    A linear layer augmented with a LoRA adapter.
    The base weight W0 is frozen; only A and B are trained.
    """
    def __init__(self, in_features, out_features, rank=4, alpha=1.0, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        # Frozen pretrained weight
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self.linear.weight.requires_grad = False  # freeze base weight
        if bias and self.linear.bias is not None:
            self.linear.bias.requires_grad = False

        # LoRA adapters: A and B
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))   # B=0 at init

    def forward(self, x):
        # Base forward (frozen)
        base_out = self.linear(x)

        # LoRA delta: x @ A^T @ B^T * scaling
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T * self.scaling

        return base_out + lora_out

    def merge_weights(self):
        """Merge LoRA into base weight for zero-overhead inference."""
        with torch.no_grad():
            self.linear.weight += (self.lora_B @ self.lora_A) * self.scaling
        self.lora_A.requires_grad = False
        self.lora_B.requires_grad = False

    def trainable_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def total_params(self):
        return sum(p.numel() for p in self.parameters())


# Instantiate and inspect
d_in, d_out = 768, 768
lora_layer = LoRALinear(d_in, d_out, rank=8, alpha=8.0)

total    = lora_layer.total_params()
trainable = lora_layer.trainable_params()
frozen   = total - trainable

print(f"Layer: Linear({d_in}, {d_out}) + LoRA(r=8)")
print(f"  Total params:     {total:,}")
print(f"  Trainable (LoRA): {trainable:,}")
print(f"  Frozen (base):    {frozen:,}")
print(f"  LoRA % of total:  {trainable/total*100:.2f}%")

# Compare across ranks
print("\nLoRA parameter counts vs full fine-tuning:")
print(f"  Full fine-tuning: {d_in * d_out:,} params")
for r in [1, 2, 4, 8, 16, 32, 64]:
    lora_params = r * (d_in + d_out)
    pct = lora_params / (d_in * d_out) * 100
    print(f"  r={r:2d}: {lora_params:,} params  ({pct:.2f}%)")

# Forward pass test
x = torch.randn(2, 10, d_in)   # (B, T, d_in)
y = lora_layer(x)
print(f"\nForward pass: input {x.shape} -> output {y.shape}")

# Verify B=0 initialization means delta = 0 at init
with torch.no_grad():
    base_out = lora_layer.linear(x)
    delta = y - base_out
    print(f"LoRA delta norm at init (B=0): {delta.norm():.2e}  (should be ~0)")

## Summary

This chapter covered the linear algebra foundations that power every LLM:

| Concept | LLM Usage |
|---|---|
| Vectors & cosine similarity | Embedding comparison, retrieval |
| Matrix multiplication | Linear layers, attention projections |
| Rank-3 tensors $(B, T, d)$ | Batched sequence processing |
| Eigendecomposition | Weight analysis, initialization theory |
| SVD & low-rank approximation | LoRA, model compression |
| Embedding lookup | Token-to-vector conversion |
| Scaled dot-product attention | Transformer core mechanism |
| LoRA linear layer | Parameter-efficient fine-tuning |

**Key takeaways:**
- The scaling factor $1/\sqrt{d_k}$ in attention is essential for gradient stability
- LoRA works because weight updates in fine-tuning have low intrinsic rank (SVD intuition)
- Everything in a transformer is differentiable linear algebra — enabling end-to-end backprop

**Next:** Chapter 2 covers Calculus & Optimization — how we train LLMs with backpropagation and adaptive optimizers.